In [ ]:
!pip install -q transformers peft accelerate bitsandbytes sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "CohereForAI/aya-expanse-8b"

# CHANGE THIS
ADAPTER_PATH = "/kaggle/input/wmt-zh-en-lora"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loading adapter...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

print("Merging LoRA weights...")
merged_model = model.merge_and_unload()

merged_model.eval()

print("✅ Final finetuned model ready")

In [ ]:
import tarfile
import urllib.request
import tempfile
import os

samples_to_eval = 100

sources = []
references = []

with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:

    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz",
        tmp.name
    )

    with tarfile.open(tmp.name, "r:gz") as tar:

        eng_file = tar.extractfile(
            "./flores200_dataset/dev/eng_Latn.dev"
        )

        zho_file = tar.extractfile(
            "./flores200_dataset/dev/zho_Hans.dev"
        )

        sources = [
            line.decode("utf-8").strip()
            for line in eng_file.readlines()
        ][:samples_to_eval]

        references = [
            line.decode("utf-8").strip()
            for line in zho_file.readlines()
        ][:samples_to_eval]

os.remove(tmp.name)

print("Loaded", len(sources), "FLORES samples") 

In [ ]:
from tqdm.auto import tqdm

predictions = []

for src_text in tqdm(sources):

    prompt = (
        f"Translate from English to Simplified Chinese:\n"
        f"en: {src_text}\n"
        f"zh:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(merged_model.device)

    with torch.no_grad():

        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    pred = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    predictions.append(pred)

print("Done")

In [ ]:
!pip install -q evaluate unbabel-comet pytorch-lightning "transformers<4.45"

import json

data = [
    {
        "src": src,
        "mt": mt,
        "ref": ref
    }
    for src, mt, ref in zip(
        sources,
        predictions,
        references
    )
]

with open("data_to_grade.json", "w", encoding="utf-8") as f:
    json.dump(data, f)

In [ ]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")

comet_model = load_from_checkpoint(model_path)

results = comet_model.predict(
    data,
    batch_size=8,
    gpus=1
)

print("=" * 50)
print("COMET SCORE:", results.system_score)
print("=" * 50)